[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/google-genai/routing-agent.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# Google GenAI SDK - 构建路由智能体

## 安装依赖

In [ ]:
!pip install -q google-genai arize-phoenix-otel openinference-instrumentation-google-genai

## 连接 Arize Phoenix

In [ ]:
import os
from getpass import getpass

from google import genai
from phoenix.otel import register

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

tracer_provider = register(auto_instrument=False, project_name="google-genai-routing-agent")
tracer = tracer_provider.get_tracer(__name__)

## 通过 Google Vertex AI 进行身份验证

In [ ]:
!gcloud auth login

In [ ]:
from google.genai import types

In [ ]:
# 使用 Vertex AI API 创建客户端，此处也可以改用 Google GenAI API
client = genai.Client(vertexai=True, project="<填写 GCP 项目 ID>", location="us-central1")

# 路由

In [ ]:
@tracer.tool()
def get_current_weather(location: str) -> str:
    """返回当前天气。

    Args:
      location: 城市和州，例如 San Francisco, CA
    """
    return "sunny"


@tracer.tool()
def get_current_time(location: str) -> str:
    """返回当前时间。

    Args:
      location: 城市和州，例如 San Francisco, CA
    """
    return "10:00 AM"


@tracer.llm()
def call_google_genai(query: str):
    response = client.models.generate_content(
        model="gemini-2.0-flash-001",
        contents=query,
        config=types.GenerateContentConfig(tools=[get_current_weather, get_current_time]),
    )
    return response.text


print(call_google_genai(query="What is the weather like in Boston?"))
print(call_google_genai(query="What is the time in Boston?"))